# Phase 2: BiLSTM-CRF baseline (Hindi)

Owner: Sajan Kumar. Plan: [docs/01-roadmap.md](../docs/01-roadmap.md), Phase 2.

**Model** (Lample et al. 2016): each word becomes a fastText Hindi vector plus a character-level BiLSTM feature (which helps with rare and unseen words). A word-level BiLSTM reads the whole sentence. A **CRF** layer then picks the best *sequence* of tags, using learned tag-to-tag transition scores.

**Gate:** the Hindi gold entity F1 is logged in `results/results.csv`. The expected range is mid-to-high 60s. This is a baseline, so don't chase SOTA.

**How to run**
1. Open this notebook in **Colab** (dev/debug) or **Kaggle** (long run) with a **GPU** runtime.
2. Run it once with `DEBUG = True`: 5k sentences and 2 epochs, finishing in a few minutes, to confirm the pipeline works end to end.
3. Set `DEBUG = False` and *Run all*. Checkpoints are saved every epoch to Drive or Kaggle. If the session dies, just *Run all* again and it resumes.
4. Copy the logged row into `results/results.csv` in the repo and commit it (see the last cell).

Shared code comes from `src/` (loader, label map, seqeval evaluator, results logger). Do **not** redefine any of it here.

In [1]:
# Pin versions (docs/04-compute.md). Colab/Kaggle already ship torch + numpy.
%pip install -q pytorch-crf==0.7.2 seqeval==1.2.2

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/iPriyadarshi/NaamRekha.git"

# Local: the notebook runs from notebooks/, so the repo root is "..".
# Colab/Kaggle: clone the repo next to the notebook.
if os.path.isdir("../src"):
    REPO_ROOT = os.path.abspath("..")
elif os.path.isdir("src"):
    REPO_ROOT = os.path.abspath(".")
else:
    REPO_ROOT = os.path.abspath("NaamRekha")
    if not os.path.isdir(REPO_ROOT):
        # If the repo is private, use https://<token>@github.com/... or upload src/ manually.
        url = REPO_URL
        token = None
        try:
            from google.colab import userdata
            token = userdata.get("GH_TOKEN")      # Colab Secret, if available
        except Exception:
            pass
        if not token:
            from getpass import getpass
            token = getpass("GitHub token (Enter to skip if repo is public): ")
        if token:
            url = REPO_URL.replace("https://", f"https://{token}@")
        subprocess.run(["git", "clone", "--depth", "1", url, REPO_ROOT], check=True)
sys.path.insert(0, REPO_ROOT)

from src import LABELS, evaluate, id2label, label2id, load_split, log_result, tag_distribution

print("repo:", REPO_ROOT)
print("labels:", LABELS)

repo: /content/NaamRekha
labels: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']


## Config

Every knob is in this one cell. Record anything you change in the `notes` of the results row (that happens automatically for the main ones).

In [3]:
SEED = 42
LANG = "hi"

DEBUG = False             # True: 5k sentences, 2 epochs (pipeline check). False: the real run.
SUBSET_SIZE = 100_000    # training sentences sampled from the ~985k silver train set
EPOCHS = 20              # upper bound; early stopping on val F1 usually ends sooner
PATIENCE = 3             # stop after this many epochs without a val-F1 improvement

BATCH_SIZE = 32
LR = 1e-3                # Adam
CLIP = 5.0               # gradient-norm clipping
DROPOUT = 0.5
WORD_DIM = 300           # fastText cc.hi.300
CHAR_DIM = 30
CHAR_HIDDEN = 25         # per direction
HIDDEN = 128             # word BiLSTM, per direction
MAX_WORD_CHARS = 20      # words longer than this are truncated for the char BiLSTM
WORD_DROPOUT = 0.5       # chance to replace a singleton train word with <unk> (teaches <unk>)

USE_FASTTEXT = True
FASTTEXT_URL = "https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.hi.300.vec.gz"

if DEBUG:
    SUBSET_SIZE, EPOCHS = 5_000, 2

In [4]:
def pick_storage() -> str:
    # Persistent storage for checkpoints: Kaggle working dir, Google Drive, or local checkpoints/.
    if os.path.isdir("/kaggle/working"):
        return "/kaggle/working/naamrekha"
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
        return "/content/drive/MyDrive/NaamRekha"
    except ImportError:
        return os.path.join(REPO_ROOT, "checkpoints")


STORAGE = pick_storage()
RUN_NAME = f"bilstm-crf_{LANG}_n{SUBSET_SIZE}_s{SEED}" + ("_debug" if DEBUG else "")
CKPT_DIR = os.path.join(STORAGE, "phase2", RUN_NAME)
FT_DIR = os.path.join(STORAGE, "fasttext")
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(FT_DIR, exist_ok=True)
print("checkpoints ->", CKPT_DIR)

Mounted at /content/drive
checkpoints -> /content/drive/MyDrive/NaamRekha/phase2/bilstm-crf_hi_n100000_s42


In [5]:
import random

import numpy as np
import torch

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device)
if device.type == "cpu":
    print("WARNING: no GPU. Fine for DEBUG, too slow for the real run.")

torch 2.14.1+cu130 | device: cuda


## Data

Train on a seeded random **subset** of the silver train set. Pick the best epoch on **val** (also silver). Report on the **full gold test** set, and never touch test for any decision.

In [6]:
train_full = load_split(LANG, "train")
val = load_split(LANG, "val")
test = load_split(LANG, "test")

n = min(SUBSET_SIZE, len(train_full))
idx = sorted(random.Random(SEED).sample(range(len(train_full)), n))
train = [train_full[i] for i in idx]
del train_full

print(f"train {len(train):,} (subset) | val {len(val):,} | test {len(test):,}")


def entity_report(rows, name):
    # Entity density check (docs/03-conventions.md): make sure the subset isn't mostly O.
    counts = tag_distribution(rows)
    total = sum(counts.values())
    ents = {t: counts.get(f"B-{t}", 0) for t in ("PER", "ORG", "LOC")}
    with_ent = sum(any(t != "O" for t in tags) for _, tags in rows)
    print(f"{name}: entities {ents} | O = {counts['O'] / total:.1%} of tokens | "
          f"{with_ent / len(rows):.1%} of sentences have >=1 entity")


entity_report(train, "train subset")
entity_report(test, "gold test")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


data/hi_IndicNER_v1.0.zip: reconstructing file:   0%|          |  0.00B / 82.3MB            

data/hi_IndicNER_v1.0.zip: downloading bytes:           |  0.00B            

train 100,000 (subset) | val 13,460 | test 864
train subset: entities {'PER': 78044, 'ORG': 70041, 'LOC': 74368} | O = 82.0% of tokens | 90.6% of sentences have >=1 entity
gold test: entities {'PER': 788, 'ORG': 521, 'LOC': 613} | O = 83.0% of tokens | 86.0% of sentences have >=1 entity


## Vocabulary and fastText embeddings

- Word vocab = every word in the train subset, plus val/test words that have a fastText vector. Pretrained vectors are unsupervised, so using them for test words is standard practice (Lample et al. do the same) and leaks no labels.
- Words with no fastText vector start random and are learned.
- The full `.vec.gz` is about 1.2 GB. It is downloaded once to a local cache, and only the vectors we need are saved to persistent storage.

In [7]:
import gzip
import urllib.request
from collections import Counter

from tqdm.auto import tqdm

PAD, UNK = "<pad>", "<unk>"

word_counts = Counter(w for words, _ in train for w in words)
wanted = set(word_counts) | {w for rows in (val, test) for words, _ in rows for w in words}


def load_fasttext_subset(wanted_words: set) -> dict:
    cache = os.path.join(FT_DIR, f"cc.{LANG}.300.filtered_{RUN_NAME}.pt")
    if os.path.exists(cache):
        print("fastText: using cached subset", cache)
        return torch.load(cache)
    raw = os.path.join(os.path.expanduser("~/.cache/naamrekha"), os.path.basename(FASTTEXT_URL))
    os.makedirs(os.path.dirname(raw), exist_ok=True)
    if not os.path.exists(raw):
        print("fastText: downloading", FASTTEXT_URL, "(~1.2 GB, one time)")
        urllib.request.urlretrieve(FASTTEXT_URL, raw)
    vecs = {}
    with gzip.open(raw, "rt", encoding="utf-8") as f:
        total, _dim = map(int, f.readline().split())
        for line in tqdm(f, total=total, desc="scan fastText"):
            word, rest = line.rstrip("\n").split(" ", 1)
            if word in wanted_words:
                vecs[word] = torch.tensor(np.asarray(rest.split(), dtype=np.float32))
    torch.save(vecs, cache)
    return vecs


ft = load_fasttext_subset(wanted) if USE_FASTTEXT else {}

# Deterministic order: by frequency, then alphabetically.
train_words = sorted(word_counts, key=lambda w: (-word_counts[w], w))
extra_words = sorted(w for w in wanted - set(word_counts) if w in ft)
itos = [PAD, UNK] + train_words + extra_words
stoi = {w: i for i, w in enumerate(itos)}

emb = torch.empty(len(itos), WORD_DIM).normal_(0, 0.1, generator=torch.Generator().manual_seed(SEED))
emb[0] = 0.0
hits = 0
for w, i in stoi.items():
    if w in ft:
        emb[i] = ft[w]
        hits += 1

singleton_ids = torch.tensor([stoi[w] for w, c in word_counts.items() if c == 1])

char_counts = Counter(c for w in word_counts for c in w)
chars = [PAD, UNK] + sorted(char_counts, key=lambda c: (-char_counts[c], c))
ctoi = {c: i for i, c in enumerate(chars)}


def oov_rate(rows):
    toks = [w for words, _ in rows for w in words]
    return sum(w not in stoi for w in toks) / len(toks)


print(f"vocab {len(itos):,} words ({hits:,} with fastText = {hits / len(itos):.1%}) | {len(chars)} chars")
print(f"test token OOV rate: {oov_rate(test):.2%}")

fastText: using cached subset /content/drive/MyDrive/NaamRekha/fasttext/cc.hi.300.filtered_bilstm-crf_hi_n100000_s42.pt
vocab 98,952 words (77,233 with fastText = 78.1%) | 555 chars
test token OOV rate: 0.73%


## Batching

In [8]:
from torch.utils.data import DataLoader


def encode(words, tags):
    w = [stoi.get(x, 1) for x in words]
    c = [[ctoi.get(ch, 1) for ch in x[:MAX_WORD_CHARS]] for x in words]
    t = [label2id[x] for x in tags]
    return w, c, t


def collate(batch):
    B = len(batch)
    T = max(len(w) for w, _, _ in batch)
    L = max(len(cw) for _, c, _ in batch for cw in c)
    words = torch.zeros(B, T, dtype=torch.long)
    tags = torch.zeros(B, T, dtype=torch.long)
    mask = torch.zeros(B, T, dtype=torch.bool)
    chars_t = torch.zeros(B, T, L, dtype=torch.long)
    char_lens = torch.zeros(B, T, dtype=torch.long)
    for b, (w, c, t) in enumerate(batch):
        words[b, : len(w)] = torch.tensor(w)
        tags[b, : len(t)] = torch.tensor(t)
        mask[b, : len(w)] = True
        for j, cw in enumerate(c):
            chars_t[b, j, : len(cw)] = torch.tensor(cw)
            char_lens[b, j] = len(cw)
    return words, chars_t, char_lens, tags, mask


train_enc = [encode(w, t) for w, t in train]
val_enc = [encode(w, t) for w, t in val]
test_enc = [encode(w, t) for w, t in test]

## Model: char-BiLSTM + word-BiLSTM + CRF

The CRF is left **unconstrained**, so it does not forbid `O -> I-X`. The silver data contains about 0.5% orphan `I-` tags, which we keep as-is per the Phase 1 contract, so the CRF should learn the transitions from the data.

In [9]:
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torchcrf import CRF


class BiLSTMCRF(nn.Module):
    def __init__(self, word_emb: torch.Tensor, n_chars: int, n_tags: int):
        super().__init__()
        self.word_emb = nn.Embedding.from_pretrained(word_emb, freeze=False, padding_idx=0)
        self.char_emb = nn.Embedding(n_chars, CHAR_DIM, padding_idx=0)
        self.char_lstm = nn.LSTM(CHAR_DIM, CHAR_HIDDEN, batch_first=True, bidirectional=True)
        self.lstm = nn.LSTM(word_emb.size(1) + 2 * CHAR_HIDDEN, HIDDEN, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(DROPOUT)
        self.proj = nn.Linear(2 * HIDDEN, n_tags)
        self.crf = CRF(n_tags, batch_first=True)

    def char_features(self, chars, char_lens):
        # One vector per word: final forward + backward states of the char BiLSTM.
        B, T, L = chars.shape
        flat, lens = chars.view(B * T, L), char_lens.view(-1)
        out = chars.new_zeros(B * T, 2 * CHAR_HIDDEN, dtype=torch.float)
        real = lens > 0  # skip padding positions
        packed = pack_padded_sequence(self.char_emb(flat[real]), lens[real].cpu(),
                                      batch_first=True, enforce_sorted=False)
        _, (h, _) = self.char_lstm(packed)
        out[real] = torch.cat([h[0], h[1]], dim=-1)
        return out.view(B, T, -1)

    def emissions(self, words, chars, char_lens, mask):
        x = torch.cat([self.word_emb(words), self.char_features(chars, char_lens)], dim=-1)
        x = self.dropout(x)
        packed = pack_padded_sequence(x, mask.sum(1).cpu(), batch_first=True, enforce_sorted=False)
        h, _ = self.lstm(packed)
        h, _ = pad_packed_sequence(h, batch_first=True, total_length=words.size(1))
        return self.proj(self.dropout(h))

    def loss(self, words, chars, char_lens, tags, mask):
        return -self.crf(self.emissions(words, chars, char_lens, mask), tags, mask=mask, reduction="mean")

    def decode(self, words, chars, char_lens, mask):
        return self.crf.decode(self.emissions(words, chars, char_lens, mask), mask=mask)


model = BiLSTMCRF(emb, len(chars), len(LABELS)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")

30,207,032 parameters


## Train (checkpoint every epoch, resume automatically)

- `last.pt` is the state after the latest epoch (model, optimizer, history). If it exists, training resumes from it.
- `best.pt` holds the weights with the best **val** F1, plus the vocab, so it can be reused later.

In [10]:
import time


@torch.no_grad()
def predict(enc_rows, batch_size=256):
    model.eval()
    preds = []
    for words, chars_t, char_lens, _, mask in DataLoader(enc_rows, batch_size=batch_size, collate_fn=collate):
        out = model.decode(words.to(device), chars_t.to(device), char_lens.to(device), mask.to(device))
        preds.extend([id2label[i] for i in seq] for seq in out)
    return preds


def score(rows, enc_rows):
    return evaluate([tags for _, tags in rows], predict(enc_rows))


LAST, BEST = os.path.join(CKPT_DIR, "last.pt"), os.path.join(CKPT_DIR, "best.pt")
start_epoch, best_f1, best_epoch, bad_epochs, history = 0, -1.0, 0, 0, []

if os.path.exists(LAST):
    st = torch.load(LAST, map_location=device)
    model.load_state_dict(st["model"])
    optimizer.load_state_dict(st["optimizer"])
    start_epoch, best_f1, best_epoch = st["epoch"], st["best_f1"], st["best_epoch"]
    bad_epochs, history = st["bad_epochs"], st["history"]
    print(f"Resumed after epoch {start_epoch} (best val F1 {best_f1:.4f} @ epoch {best_epoch})")

singletons = singleton_ids.to(device)

for epoch in range(start_epoch + 1, EPOCHS + 1):
    if bad_epochs >= PATIENCE:
        print("Early stopping already triggered.")
        break
    model.train()
    t0, total = time.time(), 0.0
    g = torch.Generator().manual_seed(SEED + epoch)  # reproducible shuffle, even after resume
    loader = DataLoader(train_enc, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate, generator=g)
    for words, chars_t, char_lens, tags, mask in tqdm(loader, desc=f"epoch {epoch}", leave=False):
        words, chars_t, char_lens = words.to(device), chars_t.to(device), char_lens.to(device)
        tags, mask = tags.to(device), mask.to(device)
        if WORD_DROPOUT > 0:
            drop = torch.isin(words, singletons) & (torch.rand(words.shape, device=device) < WORD_DROPOUT)
            words = words.masked_fill(drop, 1)  # 1 = <unk>
        loss = model.loss(words, chars_t, char_lens, tags, mask)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CLIP)
        optimizer.step()
        total += loss.item()

    val_scores = score(val, val_enc)
    history.append({"epoch": epoch, "loss": total / len(loader), **val_scores})
    improved = val_scores["F1_overall"] > best_f1
    if improved:
        best_f1, best_epoch, bad_epochs = val_scores["F1_overall"], epoch, 0
        torch.save({"model": model.state_dict(), "itos": itos, "chars": chars, "labels": LABELS,
                    "epoch": epoch, "val_scores": val_scores}, BEST)
    else:
        bad_epochs += 1
    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(), "epoch": epoch,
                "best_f1": best_f1, "best_epoch": best_epoch, "bad_epochs": bad_epochs,
                "history": history}, LAST)
    print(f"epoch {epoch:2d} | loss {total / len(loader):.3f} | val F1 {val_scores['F1_overall']:.4f}"
          f"{' *' if improved else ''} | {time.time() - t0:.0f}s")

Resumed after epoch 1 (best val F1 0.7424 @ epoch 1)


epoch 2:   0%|          | 0/3125 [00:00<?, ?it/s]

epoch  2 | loss 2.973 | val F1 0.7484 * | 372s


epoch 3:   0%|          | 0/3125 [00:00<?, ?it/s]

epoch  3 | loss 2.566 | val F1 0.7517 * | 386s


epoch 4:   0%|          | 0/3125 [00:00<?, ?it/s]

... (3 outputs left out)

epoch 6:   0%|          | 0/3125 [00:00<?, ?it/s]

epoch  6 | loss 1.937 | val F1 0.7402 | 395s


epoch 7:   0%|          | 0/3125 [00:00<?, ?it/s]

epoch  7 | loss 1.803 | val F1 0.7447 | 397s
Early stopping already triggered.


## Evaluate the best checkpoint on Hindi gold test

In [11]:
from seqeval.metrics import classification_report

model.load_state_dict(torch.load(BEST, map_location=device)["model"])
val_scores = score(val, val_enc)
test_preds = predict(test_enc)
test_scores = evaluate([tags for _, tags in test], test_preds)

print(f"best epoch {best_epoch} | val (silver) F1 {val_scores['F1_overall']:.4f}")
print("TEST (gold):", {k: round(v, 4) for k, v in test_scores.items()})
print(classification_report([tags for _, tags in test], test_preds, digits=4))

best epoch 4 | val (silver) F1 0.7546
TEST (gold): {'F1_overall': 0.8038, 'F1_PER': 0.8692, 'F1_ORG': 0.713, 'F1_LOC': 0.7933}
              precision    recall  f1-score   support

         LOC     0.8123    0.7752    0.7933       614
         ORG     0.7448    0.6838    0.7130       525
         PER     0.8638    0.8747    0.8692       790

   micro avg     0.8169    0.7911    0.8038      1929
   macro avg     0.8069    0.7779    0.7918      1929
weighted avg     0.8150    0.7911    0.8025      1929



## Log the run

This appends **one** row to `results/results.csv`, so run it once per real run. On Colab/Kaggle the file lives inside the cloned repo. Copy the printed row into your local `results/results.csv` and commit it, along with the notebook.

In [12]:
notes = (f"seed={SEED}; best_epoch={best_epoch}; val_F1={val_scores['F1_overall']:.4f}; "
         f"emb={'fastText cc.hi.300' if USE_FASTTEXT else 'random'}+charBiLSTM; hidden={HIDDEN}; "
         f"lr={LR}; bs={BATCH_SIZE}; dropout={DROPOUT}; word_dropout={WORD_DROPOUT}"
         + ("; DEBUG run" if DEBUG else ""))

log_result(
    model="BiLSTM-CRF",
    train_lang=LANG,
    test_lang=LANG,
    subset_size=len(train),
    epochs=len(history),
    scores=test_scores,
    checkpoint_path=BEST,
    notes=notes,
)

with open(os.path.join(REPO_ROOT, "results", "results.csv"), encoding="utf-8") as f:
    print(f.read().splitlines()[-1])

BiLSTM-CRF,hi,hi,100000,7,0.8691823899371068,0.7130089374379344,0.7933333333333333,0.8037924677376876,/content/drive/MyDrive/NaamRekha/phase2/bilstm-crf_hi_n100000_s42/best.pt,seed=42; best_epoch=4; val_F1=0.7546; emb=fastText cc.hi.300+charBiLSTM; hidden=128; lr=0.001; bs=32; dropout=0.5; word_dropout=0.5
